In [ ]:
pip install -q pypdf openai chromadb

2. Configure Open API Key

In [ ]:
import os

from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"]=getpass("Enter your OpenAI API Key:")
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("OpenAI client configured successfully.")

OpenAI client configured successfully.


3.Upload the PDF

In [ ]:
from google.colab import files
uploaded = files.upload()

pdf_path = next(iter(uploaded.keys()))
print("uploaded file:",pdf_path)


Saving CQ_Emp_Handbook.docx to CQ_Emp_Handbook.docx
uploaded file: CQ_Emp_Handbook.docx


4.PDF Reader

In [ ]:
from docx import Document

doc = Document(pdf_path)

paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
text = "\n".join(paragraphs)

print("Paragraphs:", len(paragraphs))
print("Characters extracted:", len(text))
print("\nPreview:\n")
print(text[:2000])

Paragraphs: 369
Characters extracted: 46416

Preview:

CLEARQUOTE TECHNOLOGIES INDIA PRIVATE LIMITED
Human Resources Policy Division
EMPLOYEE HANDBOOK
Policies, Conduct & Workplace Guidelines
Version 3.2  |  Effective 01 January 2026
Document Classification: Internal — All Employees
Issued by: People & Culture Team
Table of Contents
(Updates to field codes may be required if opened in some viewers — right-click and 'Update Field' in Word.)
1. Welcome & Purpose of This Handbook
Welcome to ClearQuote Technologies India Private Limited ("the Company", "ClearQuote"). This handbook has been prepared to give every employee a clear, consistent reference for the policies, expectations, and benefits that govern our workplace. It applies to all full-time, part-time, fixed-term, and contract staff across every office and remote location unless a specific policy states otherwise.
This handbook does not constitute an employment contract, nor does it guarantee employment for any specific duration. T

In [ ]:
!pip install python-docx --quiet

5.Chunking

for this example , we use simple character - based chunking startegy. In a production system , chunking can be a made more sopisticated using paragraphs , sections, headings , token counts or semantic boundaries.

In [ ]:
chunk_size = 500
overlap = 50
chunks=[]
start=0
chunk_id=0

while start < len(text):
    end= start + chunk_size
    chunk_text =text[start:end].strip()

    if chunk_text:
      chunks.append({
            "id": f"chunk-{chunk_id}",
            "text":chunk_text
      })
      chunk_id +=1
    start +=chunk_size- overlap
print("Total chunks",len(chunks))
print("\nfirst chunk:\n")
print(chunks[0]["text"])

Total chunks 104

first chunk:

CLEARQUOTE TECHNOLOGIES INDIA PRIVATE LIMITED
Human Resources Policy Division
EMPLOYEE HANDBOOK
Policies, Conduct & Workplace Guidelines
Version 3.2  |  Effective 01 January 2026
Document Classification: Internal — All Employees
Issued by: People & Culture Team
Table of Contents
(Updates to field codes may be required if opened in some viewers — right-click and 'Update Field' in Word.)
1. Welcome & Purpose of This Handbook
Welcome to ClearQuote Technologies India Private Limited ("the Company",


Embeddings

Embeddings convert each chunk of text into a numerical vector that represents its semantic meaning

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="YOUR_OPENAI_API_KEY")  # get a real OpenAI key from platform.openai.com/account/api-keys

In [ ]:
import os
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"

def get_embeddings(texts, batch_size=100):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=batch)

        ordered = sorted(response.data, key=lambda item: item.index)
        all_embeddings.extend([item.embedding for item in ordered])

    return all_embeddings

In [ ]:
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embeddings(texts, batch_size=100):
    embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True)
    return embeddings.tolist()

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_embeddings = get_embeddings(chunk_texts)

print("Number of embeddings:", len(chunk_embeddings))
print("Embedding dimensions:", len(chunk_embeddings[0]))
print("First 10 values:", chunk_embeddings[0][:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Number of embeddings: 104
Embedding dimensions: 384
First 10 values: [-0.11048657447099686, 0.02457260899245739, -0.013670844957232475, 0.013646617531776428, 0.021023720502853394, 0.03272435441613197, 0.12941473722457886, -0.07625427097082138, -0.01652423106133938, 0.01946328580379486]


7. Store in the Vector Database

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="cq_emp_handbook")

existing = collection.get()
if existing["ids"]:
    collection.delete(ids=existing["ids"])

collection.add(
    ids=[chunk["id"] for chunk in chunks],
    documents=[chunk["text"] for chunk in chunks],
    embeddings=chunk_embeddings,
    metadatas=[{"source": pdf_path} for _ in chunks]
)

print("Chunks stored in Vector Database:", collection.count())

Chunks stored in Vector Database: 104


8. User Query -> Query Embedding

In [ ]:
question ="How many casual leaves are allowance"
question_embedding = get_embeddings([question])[0]
print("Question:",question)
print("Query embedding dimensions:",len(question_embedding))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Question: How many casual leaves are allowance
Query embedding dimensions: 384


9. Retriever -> Search the Vector Database

In [ ]:
results= collection.query (query_embeddings=[question_embedding],n_results=3)

retrieved_chunks = results["documents"][0]
retrieved_ids=results["ids"][0]
distances = results["distances"][0]
print("Retrieved chunks:\n")

for i,(chunk_id,chunk_text,distance) in enumerate(zip(retrieved_ids,retrieved_chunks,distances),start=1):
    print(f"---Result{i} | {chunk_id}| distance=distance={distance:4f}---")
    print(chunk_text[:1000])
    print()

Retrieved chunks:

---Result1 | chunk-44| distance=distance=1.260678---
exploratory engineering work.
Conference sponsorship for accepted speakers/authors at recognised industry conferences.
13.4 Additional Perks
One-time home-office setup allowance of ₹15,000 for remote/hybrid employees.
Employee referral bonus for successful technical hires.
Cab reimbursement for employees working past 9:00 PM or on approved weekend shifts.
Parental transition support, including a phased return-to-work option following parental leave.
14. Travel & Expense Policy
14.1 Booking Trav

---Result2 | chunk-24| distance=distance=1.297991---
the portal within 48 hours.
Leave exceeding 5 consecutive working days requires an additional sign-off from the department head.
Unapproved absence beyond the entitled balance will be recorded as Loss of Pay (LOP) and deducted pro-rata from the monthly salary, as illustrated in the attendance and payslip examples referenced by Finance.
7.4 Public Holidays
The Company obse

In [ ]:
results

{'ids': [['chunk-44', 'chunk-24', 'chunk-43']],
 'embeddings': None,
 'documents': [['exploratory engineering work.\nConference sponsorship for accepted speakers/authors at recognised industry conferences.\n13.4 Additional Perks\nOne-time home-office setup allowance of ₹15,000 for remote/hybrid employees.\nEmployee referral bonus for successful technical hires.\nCab reimbursement for employees working past 9:00 PM or on approved weekend shifts.\nParental transition support, including a phased return-to-work option following parental leave.\n14. Travel & Expense Policy\n14.1 Booking Trav',
   'the portal within 48 hours.\nLeave exceeding 5 consecutive working days requires an additional sign-off from the department head.\nUnapproved absence beyond the entitled balance will be recorded as Loss of Pay (LOP) and deducted pro-rata from the monthly salary, as illustrated in the attendance and payslip examples referenced by Finance.\n7.4 Public Holidays\nThe Company observes a minimum of 10 d

10.Prompt + Retrieved Context

Now we combine the user's question with the relevant chunks returned by the retriever.

The retrieved context becomes part of the input sent to the LLM.

In [ ]:
context ="\n\n---- Retrived chunk---\n\n".join(retrieved_chunks)
prompt=f"""Answer the user's question using Only the retrived context below
If the answer is not the present in the context , say that the information is not available in the provided document.
Retrieved Context:{context}
User Question :{question}""".strip()

print(prompt)

Answer the user's question using Only the retrived context below
If the answer is not the present in the context , say that the information is not available in the provided document.
Retrieved Context:exploratory engineering work.
Conference sponsorship for accepted speakers/authors at recognised industry conferences.
13.4 Additional Perks
One-time home-office setup allowance of ₹15,000 for remote/hybrid employees.
Employee referral bonus for successful technical hires.
Cab reimbursement for employees working past 9:00 PM or on approved weekend shifts.
Parental transition support, including a phased return-to-work option following parental leave.
14. Travel & Expense Policy
14.1 Booking Trav

---- Retrived chunk---

the portal within 48 hours.
Leave exceeding 5 consecutive working days requires an additional sign-off from the department head.
Unapproved absence beyond the entitled balance will be recorded as Loss of Pay (LOP) and deducted pro-rata from the monthly salary, as illustrate

11. Generate the Answer -- OpenAPI Responses API


In [ ]:
from groq import Groq

groq_client = Groq(api_key="YOUR_GROQ_API_KEY")

In [ ]:
!pip install groq --quiet

In [ ]:
print(repr(groq_client.api_key))

'YOUR_GROQ_API_KEY'


In [ ]:
from groq import Groq

groq_client = Groq(api_key="YOUR_GROQ_API_KEY")

print(repr(groq_client.api_key))  # sanity check — must start with 'gsk_', not 'YOUR_OPENAI_API_KEY'

'YOUR_GROQ_API_KEY'


In [ ]:
from groq import Groq

groq_client = Groq(api_key="YOUR_GROQ_API_KEY")

In [ ]:
print(repr(groq_client.api_key))

'YOUR_GROQ_API_KEY'


In [ ]:
from google.colab import userdata
from groq import Groq

groq_client = Groq(api_key=userdata.get('GROQ_API_KEY_CQ'))

In [ ]:
GENERATION_MODEL = "openai/gpt-oss-20b"     # currently active, good quality


In [ ]:
import requests

headers = {"Authorization": f"Bearer {groq_client.api_key}"}
resp = requests.get("https://api.groq.com/openai/v1/models", headers=headers)
for m in resp.json()["data"]:
    print(m["id"])

canopylabs/orpheus-arabic-saudi
openai/gpt-oss-20b
groq/compound
allam-2-7b
canopylabs/orpheus-v1-english
groq/compound-mini
whisper-large-v3
qwen/qwen3.6-27b
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
qwen/qwen3.8-27b
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m


In [ ]:
GENERATION_MODEL = "llama-3.1-8b-instant"   # fast, currently active

In [ ]:
import requests

headers = {"Authorization": f"Bearer {groq_client.api_key}"}
resp = requests.get("https://api.groq.com/openai/v1/models", headers=headers)
print(resp.status_code)
print(resp.json())

200
{'object': 'list', 'data': [{'id': 'meta-llama/llama-prompt-guard-2-22m', 'object': 'model', 'created': 1748632101, 'owned_by': 'Meta', 'active': True, 'context_window': 512, 'public_apps': None, 'max_completion_tokens': 512, 'name': 'Llama Prompt Guard 2 22M', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 512, 'max_output_length': 512, 'pricing': {'prompt': '0.00000003', 'completion': '0.00000003', 'image': '0', 'request': '0', 'input_cache_read': '0.000000015'}, 'supported_sampling_parameters': ['temperature', 'top_p', 'stop', 'seed', 'max_tokens']}, {'id': 'openai/gpt-oss-120b', 'object': 'model', 'created': 1754408224, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536, 'hugging_face_id': 'openai/gpt-oss-120b', 'name': 'GPT OSS 120B', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 131072, 'max_output_length': 65536, 'pricing': {'prompt': '0.00000015'

In [ ]:
for m in resp.json()["data"]:
    print(m["id"])

meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
groq/compound-mini
canopylabs/orpheus-arabic-saudi
whisper-large-v3-turbo
whisper-large-v3
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
qwen/qwen3.6-27b
meta-llama/llama-prompt-guard-2-86m
allam-2-7b
openai/gpt-oss-safeguard-20b
openai/gpt-oss-20b
groq/compound


In [ ]:
GENERATION_MODEL = "openai/gpt-oss-120b"

response = groq_client.chat.completions.create(
    model=GENERATION_MODEL,
    messages=[{"role": "user", "content": prompt}]
)
answer = response.choices[0].message.content
print("Answer:\n")
print(answer)

Answer:

The information about the number of casual leaves allowed is not available in the provided document.
